# gu model
radiation effect of smoke at harvard forest from our regression and from the gu et al. 2003 canopy model with its 1992 parameters and refit to the tower (figure 4)

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.api as sm

data, out = Path("data"), Path("results")
out.mkdir(exist_ok=True)
drivers = ["PAR", "DF", "COSZ", "VPD", "TA", "SWC"]


def design(v):
    x = pd.DataFrame({k: np.asarray(v[k], float) for k in drivers})
    x["PAR"] = x.PAR / 1000
    x["PAR2"] = x.PAR ** 2
    x["PARxDF"] = x.PAR * x.DF
    return sm.add_constant(x[["PAR", "PAR2", "DF", "PARxDF", "COSZ", "VPD", "TA", "SWC"]], has_constant="add")
from scipy.optimize import curve_fit

d = pd.read_csv(data / "harvard_unfiltered.csv", parse_dates=["date"])
weather = ["VPD", "TA", "SWC", "COSZ"]

In [2]:
def gu(p, dif, dirh):
    # gu et al. 2003 som eq 2, shortwave diffuse and direct horizontal
    ad, ab, bd, bb = p
    dif, dirh = np.maximum(dif, 0), np.maximum(dirh, 0)
    a, b = ad * dif + ab * dirh, bd * dif + bb * dirh
    return a * b / np.maximum(b + a * (dif + dirh), 1e-9)


def gu_fit(s, wx=False):
    # nonlinear least squares, optionally with additive linear weather terms
    z = s[weather].values.T if wx else np.empty((0, len(s)))
    f = lambda X, ad, ab, bd, bb, *c: gu((ad, ab, bd, bb), X[0], X[1]) + (c[0] + X[2:].T @ np.array(c[1:]) if wx else 0)
    X = np.vstack([s.SW_IN * s.DF, s.SW_IN * (1 - s.DF), z])
    n = 1 + len(weather) if wx else 0
    p0 = ([0.08, 0.06, 70, 25] if wx else [0.1, 0.04, 100, 20]) + [0] * n
    lo, hi = [1e-4, 1e-4, 1, 1] + [-np.inf] * n, [1, 1, 1000, 1000] + [np.inf] * n
    p, _ = curve_fit(f, X, s.GPP.values, p0=p0, bounds=(lo, hi), maxfev=100000)
    return tuple(p[:4])

In [3]:
params = {"gu_default": (0.1182, 0.0394, 139.60, 19.33),  # gu et al. 2003 table 1, 1992
          "gu_clear": gu_fit(d[d.cloudfree]), "gu_partly": gu_fit(d), "gu_wx": gu_fit(d, wx=True)}
fit = sm.OLS(d.GPP.values, design(d)).fit()

ref = d[d.is_bg]
rows = []
for r in d[d.is_smoke].itertuples():
    w = ref[((ref.date - r.date).dt.days.abs() <= 15) & ((ref.hour - r.hour).abs() <= 1)]
    if len(w) < 3:
        continue
    s = {v: getattr(r, v) for v in drivers}
    reg = fit.predict(design({k: [v] for k, v in s.items()})).iloc[0] - \
        fit.predict(design({**{k: [v] for k, v in s.items()}, "PAR": [w.PAR.mean()], "DF": [w.DF.mean()]})).iloc[0]
    rec = {"date": r.date, "GPP_r": w.GPP.mean(), "regression": reg}
    for k, p in params.items():
        rec[k + "_s"] = gu(p, r.SW_IN * r.DF, r.SW_IN * (1 - r.DF))
        rec[k + "_r"] = gu(p, w.SW_IN * w.DF, w.SW_IN * (1 - w.DF)).mean()
    rows.append(rec)
m = pd.DataFrame(rows)

In [4]:
# change per smoke day and over all smoke hours; with weather terms the change is relative to observed gpp
pct = lambda t, k: 100 * ((t[k + "_s"] - t[k + "_r"]).sum() / (t.GPP_r if k == "gu_wx" else t[k + "_r"]).sum())
g = m.groupby("date")
days = pd.DataFrame({"regression": 100 * g.regression.sum() / g.GPP_r.sum(),
                     **{k: g.apply(lambda t, k=k: pct(t, k)) for k in params}})
summary = pd.Series({"regression": 100 * m.regression.sum() / m.GPP_r.sum(), **{k: pct(m, k) for k in params}}, name="pct")
days.to_csv(out / "gu_days.csv")
summary.to_csv(out / "gu_summary.csv", index_label="model")
summary.round(1)

Out[4]: 
regression     1.9
gu_default    23.3
gu_clear      13.2
gu_partly      8.8
gu_wx          1.2
Name: pct, dtype: float64


regression     1.9
gu_default    23.3
gu_clear      13.2
gu_partly      8.8
gu_wx          1.2
Name: pct, dtype: float64